# Kinetiq Studio — Google Colab GPU Worker (L4 / A100)
Pull-based ComfyUI worker using Colab Secrets (`API_BASE_URL`, `WORKER_TOKEN`, optional `HF_TOKEN`).
Uploads finished 1080p/4K masters and 540p proxies directly to your Google Drive via 8 MiB resumable chunks.

In [ ]:
# Cell 1: Load Secrets & Auto-Tune VRAM Flags
import os
from urllib.parse import urlsplit

import torch
from google.colab import userdata


def setup_secret(name, prefix, hint):
    try:
        value = (userdata.get(name) or '').strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise ValueError(f"Open Colab Secrets (key icon), add {name}, and enable notebook access. {hint}") from None
    if not value.startswith(prefix) or len(value) == len(prefix):
        raise ValueError(f"{name} must start with {prefix}. {hint}")
    return value

api_url = setup_secret('API_BASE_URL', 'https://', 'Use your HTTPS tunnel or VPS API origin.')
parts = urlsplit(api_url)
if not parts.hostname or parts.username or parts.password or parts.query or parts.fragment:
    raise ValueError('API_BASE_URL must be the HTTPS address of your Kinetiq API.')
os.environ['API_BASE_URL'] = api_url.rstrip('/')
os.environ['WORKER_TOKEN'] = setup_secret('WORKER_TOKEN', 'wrk_', 'Create a GPU worker token in Studio Workers.')
try:
    hf_token = (userdata.get('HF_TOKEN') or '').strip()
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    pass  # HF_TOKEN can also be set directly in the runtime environment.
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

PROFILE = 'ltx25_pro'  # options: 'ltx25_pro' | 'wan22_fast' | 'upscale_4k'
AUTO_FALLBACK_PROFILE = True  # Set False to keep the selected profile.

if not torch.cuda.is_available():
    raise RuntimeError('Attach an L4 or A100 GPU runtime before starting the worker.')
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
if vram_gb < 20:
    print(f'Notice: Requested L4/A100 GPU was unavailable; Colab assigned {gpu_name} ({vram_gb:.1f} GiB). '
          'Use wan22_fast on this runtime or reconnect later for an L4/A100.')
if PROFILE == 'ltx25_pro' and AUTO_FALLBACK_PROFILE:
    reasons = []
    if vram_gb < 20:
        reasons.append(f'the assigned GPU has {vram_gb:.1f} GiB (below 20 GiB)')
    if not os.environ.get('HF_TOKEN', '').strip():
        reasons.append('HF_TOKEN is missing for the gated LTX models')
    if reasons:
        PROFILE = 'wan22_fast'
        print('Switching from ltx25_pro to wan22_fast because ' + ' and '.join(reasons) + '.')
low_vram = 'L4' in gpu_name.upper().split() or vram_gb < 38
if low_vram:
    comfy_flags = ['--lowvram', '--fp8_e4m3fn-unet']
else:
    comfy_flags = ['--highvram' if vram_gb >= 70 else '--normalvram']
print(f'GPU: {gpu_name} ({vram_gb:.1f} GiB) | Flags: {comfy_flags} | Job profile: {PROFILE}')

In [ ]:
%%bash
# Cell 2: Install Headless ComfyUI, Custom Nodes & kinetiq-agent-core
set -e
pip install -q hf_transfer huggingface_hub httpx websocket-client psutil

if [ ! -d '/content/ComfyUI' ]; then
  git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
  pip install -q -r /content/ComfyUI/requirements.txt
fi

if [ ! -d /content/kinetiq-studio ]; then
  git clone --depth 1 https://github.com/TeddyQyuon/kinetiq-studio.git /content/kinetiq-studio
fi

cd /content/ComfyUI/custom_nodes
for repo in \
  'https://github.com/Lightricks/ComfyUI-LTXVideo.git' \
  'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git' \
  'https://github.com/1038lab/ComfyUI-FlashVSR.git' \
  'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git' \
  'https://github.com/city96/ComfyUI-GGUF.git'; do
  name=$(basename "$repo" .git)
  if [ ! -d "$name" ]; then
    git clone --depth 1 "$repo"
  fi
  if [ -f "$name/requirements.txt" ]; then pip install -q -r "$name/requirements.txt"; fi
done

In [ ]:
# Cell 3: Fast Profile-Aware Model Bootstrap via hf_transfer
import os
from pathlib import Path

from huggingface_hub import hf_hub_download
from huggingface_hub.errors import HfHubHTTPError

MODELS = Path('/content/ComfyUI/models')

def fetch(repo_id: str, filename: str, subdir: str, *, alias: str | None = None):
    dest = MODELS / subdir
    dest.mkdir(parents=True, exist_ok=True)
    path = Path(hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(dest), token=False))
    name = dest / (alias or path.name)
    if path != name and not name.exists():
        name.symlink_to(path)

if PROFILE == 'ltx25_pro':
    access_help = ("Accept the license at https://huggingface.co/Lightricks/LTX-2.5 and add HF_TOKEN "
                   "in Colab Secrets with notebook access, then rerun Cells 1 and 3. "
                   "Or set PROFILE = 'wan22_fast' for public models that need no token.")
    token = os.environ.get('HF_TOKEN', '').strip()
    if not token:
        raise RuntimeError('ltx25_pro requires HF_TOKEN. ' + access_help)
    for filename in (
        'text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
        'vae/ltx-2.5-video-vae-bf16.safetensors',
        'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
    ):
        try:
            hf_hub_download('Lightricks/LTX-2.5', filename, local_dir=str(MODELS), token=token)
        except HfHubHTTPError as exc:
            if exc.response.status_code in (401, 403):
                raise RuntimeError('Hugging Face denied access to the LTX models. ' + access_help) from exc
            raise
    fetch('unsloth/LTX-2.5-FP8', 'LTX-2.5-FP8.safetensors', 'diffusion_models',
          alias='ltx-2.5-22b-distilled-transformer-fp8.safetensors')

if PROFILE in {"ltx25_pro", "wan22_fast"}:
    fetch('Comfy-Org/Qwen-Image', 'split_files/diffusion_models/qwen_image_fp8_e4m3fn.safetensors', 'diffusion_models')
    fetch('Comfy-Org/Qwen-Image', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', 'text_encoders')
    fetch('Comfy-Org/Qwen-Image', 'split_files/vae/qwen_image_vae.safetensors', 'vae')

if PROFILE == 'wan22_fast':
    base = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
    fetch(base, 'split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors', 'diffusion_models')
    fetch(base, 'split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors', 'diffusion_models')
    fetch(base, 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors', 'loras')
    fetch(base, 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors', 'loras')
    fetch(base, 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 'text_encoders')
    fetch(base, 'split_files/vae/wan_2.1_vae.safetensors', 'vae')
elif PROFILE == 'upscale_4k':
    fetch('numz/SeedVR2_comfyUI', 'seedvr2_ema_3b_fp8_e4m3fn.safetensors', 'SEEDVR2')
    fetch('numz/SeedVR2_comfyUI', 'ema_vae_fp16.safetensors', 'SEEDVR2')
print('Model bootstrap complete for profile:', PROFILE)

In [ ]:
# Cell 4: Start ComfyUI before registering or leasing work
import subprocess
import time
from contextlib import suppress

import httpx

with open('/content/comfy.log', 'w') as log:
    comfy_proc = subprocess.Popen(
        ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188',
         *comfy_flags, '--disable-auto-launch', '--preview-method', 'none'],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

for _ in range(90):
    if comfy_proc.poll() is not None:
        raise RuntimeError('ComfyUI exited during startup. See /content/comfy.log.')
    with suppress(httpx.RequestError):
        response = httpx.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if response.is_success:
            break
    time.sleep(2)
else:
    raise TimeoutError('ComfyUI startup timed out. See /content/comfy.log.')

if PROFILE == 'upscale_4k':
    nodes = httpx.get('http://127.0.0.1:8188/object_info', timeout=30)
    nodes.raise_for_status()
    required = {'AILab_FlashVSR', 'SeedVR2LoadDiTModel', 'SeedVR2LoadVAEModel', 'SeedVR2VideoUpscaler',
                'VHS_LoadVideo', 'VHS_VideoInfoLoaded', 'VHS_VideoCombine'}
    if missing := required.difference(nodes.json()):
        raise RuntimeError(f'Missing upscale nodes: {sorted(missing)}. See /content/comfy.log.')
print('ComfyUI ready on 127.0.0.1:8188')

In [ ]:
# Cell 5: Start the GPU worker in the background so Cell 6 can share this runtime.
import sys
from pathlib import Path

repo = Path('/content/kinetiq-studio')
gpu_worker = globals().get('gpu_worker')
if gpu_worker is None or gpu_worker.poll() is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(repo / 'workers/agent-core')], check=True)
    with open('/content/kinetiq-gpu.log', 'w') as log:
        gpu_worker = subprocess.Popen(
            [sys.executable, '-u', str(repo / 'workers/gpu/runner.py'), '--profile', PROFILE,
             '--provider', 'google_colab', '--idle-minutes', '15'], cwd=repo, stdout=log, stderr=subprocess.STDOUT)
print('GPU worker PID:', gpu_worker.pid, '| Log: /content/kinetiq-gpu.log')

## Cell 6 (Optional): Run Media (Kokoro TTS + WhisperX) or Zero-Key Colab AI Helper on this Runtime
For media, create a **separate media worker** in Studio and save its token as `MEDIA_WORKER_TOKEN` in Colab Secrets. Media models run on CPU in their own environment. For the text helper, choose `colab_ai` and enter a prompt. Cell 5 keeps running in the background. Its 15-minute GPU idle timer applies to the whole runtime.

In [ ]:
# @title Optional media worker or Colab AI helper
mode = "skip" # @param ["skip", "media", "colab_ai"]
prompt = "Suggest a visual example for a short lesson about tides." # @param {type:"string"}

if mode == "colab_ai":
    from google.colab import ai
    print(ai.generate_text(prompt))
elif mode == "media":
    media_token = setup_secret('MEDIA_WORKER_TOKEN', 'wrk_', 'Create a separate media worker token in Studio.')
    if media_token == os.environ['WORKER_TOKEN']:
        raise ValueError('Give the media worker its own token so both workers can stay registered.')
    media_worker = globals().get('media_worker')
    if media_worker is None or media_worker.poll() is not None:
        media_env = Path('/content/kinetiq-media-env')
        subprocess.run(['bash', str(repo / 'workers/media/setup-colab.sh'), str(media_env)], cwd=repo, check=True)
        with open('/content/kinetiq-media.log', 'w') as log:
            media_worker = subprocess.Popen(
                [str(media_env / 'bin/python'), '-u', '-m', 'workers.media.runner', '--provider', 'google_colab'], cwd=repo,
                env={**os.environ, 'WORKER_TOKEN': media_token, 'MEDIA_MOCK_MODE': 'false'},
                stdout=log, stderr=subprocess.STDOUT)
    print('Media worker PID:', media_worker.pid, '| Log: /content/kinetiq-media.log')